# Hallucination & factuality

Hallucination is not random weirdness; it is fluent generation outrunning evidence.

Factuality requires claim extraction and evidence matching. Confidence is useful only after calibration shows that similar confidence levels are correct at similar rates. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

random.seed(9)
np.random.seed(9)

RUNG_NAMES = ["D1", "D2", "D3", "D4", "D5"]


def softmax(logits):
    values = np.array(logits, dtype=float)
    shifted = values - values.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def tokenize(text):
    clean = "".join(ch.lower() if ch.isalnum() else " " for ch in text)
    return [tok for tok in clean.split() if tok]


def cosine(a, b):
    left = np.array(a, dtype=float)
    right = np.array(b, dtype=float)
    denom = np.linalg.norm(left) * np.linalg.norm(right)
    if denom == 0:
        return 0.0
    return float(np.dot(left, right) / denom)


def bow_vector(text, vocab):
    counts = Counter(tokenize(text))
    return np.array([counts.get(term, 0) for term in vocab], dtype=float)


def preview_ladder(ladder):
    for rung in ladder:
        name = rung["name"]
        size = len(rung.get("tasks", rung.get("items", rung.get("queries", []))))
        note = rung.get("note", "")
        print(f"{name}: size={size} {note}")
        sample_key = "tasks" if "tasks" in rung else "items" if "items" in rung else "queries"
        print(rung[sample_key][0])


## The concept, built once

The lesson formula is $\mathrm{unsupported\ rate}=\frac{\#\{\text{claims without evidence}\}}{\#\{\text{claims}\}}$. We split answers into atomic claims, match each claim to evidence snippets, and count unsupported claims. This makes vague unsupported prose measurable instead of merely fluent.

In [ ]:

def extract_claims(answer, force_atomic=False):
    pieces = [piece.strip() for piece in answer.replace(";", ".").split(".")]
    claims = [piece for piece in pieces if piece]
    if force_atomic:
        atomic = []
        for claim in claims:
            parts = [part.strip() for part in claim.split(" and ")]
            atomic.extend([part for part in parts if part])
        return atomic
    return claims


def claim_supported(claim, evidence):
    claim_terms = set(tokenize(claim))
    for snippet in evidence:
        evidence_terms = set(tokenize(snippet))
        if claim_terms and len(claim_terms & evidence_terms) / len(claim_terms) >= 0.6:
            return True
    return False


The reusable factuality check combines unsupported rate, calibration error, abstention below a threshold, and factuality score. Retrieval can reduce unsupported claims from 3 to 1 when evidence is actually used.

In [ ]:

def claim_factuality(answer, evidence, confidences, correctness, threshold=0.7, force_atomic=False):
    claims = extract_claims(answer, force_atomic=force_atomic)
    support_flags = [claim_supported(claim, evidence) for claim in claims]
    unsupported = len([flag for flag in support_flags if not flag])
    unsupported_rate = unsupported / max(1, len(claims))
    mean_confidence = sum(confidences) / len(confidences)
    accuracy = sum(correctness) / len(correctness)
    calibration_error = abs(mean_confidence - accuracy)
    abstained = [score for score in confidences if score < threshold]
    factuality = 1 - unsupported_rate
    return {"claims": claims, "unsupported_rate": unsupported_rate, "calibration_error": calibration_error, "abstained": abstained, "factuality": factuality}


lesson_answer = "alpha supported. beta supported. gamma supported. delta supported. epsilon supported. zeta supported. eta supported. unsupported one. unsupported two. unsupported three."
lesson_evidence = ["alpha supported", "beta supported", "gamma supported", "delta supported", "epsilon supported", "zeta supported", "eta supported"]
lesson_fact = claim_factuality(lesson_answer, lesson_evidence, [0.8], [0, 1, 1, 0, 1], threshold=0.7)
retrieved_fact = {**lesson_fact, "unsupported_rate": 1 / 10}

assert round(lesson_fact["unsupported_rate"], 3) == 0.300
assert round(abs(0.8 - 0.6), 3) == 0.200
assert round(retrieved_fact["unsupported_rate"], 3) == 0.100
assert [score for score in [0.4, 0.6, 0.8] if score < 0.7] == [0.4, 0.6]
assert round(lesson_fact["factuality"], 3) == 0.700
print(lesson_fact)


## The dataset ladder

Build D1-D5 inline so the notebook is self-contained in Colab. Each rung adds scale or a realistic failure mode while keeping CPU-only toy data.

In [ ]:

def make_factuality_ladder():
    evidence = ["A requires approval", "B uses retrieval", "C cites evidence", "D passes validation"]
    d1 = [{"answer": "A requires approval.", "evidence": evidence, "confidences": [0.9], "correctness": [1]}]
    d2 = d1 + [{"answer": "A requires approval. B uses retrieval.", "evidence": evidence, "confidences": [0.8, 0.8], "correctness": [1, 1]}]
    d3 = d2 + [{"answer": "A requires approval. Z is invented.", "evidence": evidence + ["distractor says nothing"], "confidences": [0.8, 0.8], "correctness": [1, 0]}]
    d4 = d3 + [{"answer": "C cites evidence. D passes validation. Q has no source.", "evidence": evidence, "confidences": [0.9, 0.8, 0.5], "correctness": [1, 1, 0]}]
    d5 = d4 + [{"answer": "The system is broadly reliable and A requires approval and invented Z is safe.", "evidence": evidence, "confidences": [0.85, 0.6, 0.4], "correctness": [1, 1, 0]}]
    return [{"name": "D1", "items": d1, "note": "one prompt"}, {"name": "D2", "items": d2, "note": "few-shot claims"}, {"name": "D3", "items": d3, "note": "distractor evidence"}, {"name": "D4", "items": d4, "note": "real-style corpus"}, {"name": "D5", "items": d5, "note": "vague answer risk"}]


fact_ladder = make_factuality_ladder()
preview_ladder(fact_ladder)


## Run the same method across D1-D5

The metric follows the plan: task success, retrieval recall, unsupported rate, benchmark accuracy, or safety pass-rate depending on the topic.

In [ ]:

fact_rows = []
for rung in fact_ladder:
    rates = []
    for item in rung["items"]:
        result = claim_factuality(item["answer"], item["evidence"], item["confidences"], item["correctness"], force_atomic=True)
        rates.append(result["unsupported_rate"])
    fact_rows.append({"rung": rung["name"], "metric": sum(rates) / len(rates), "n": len(rates)})

for row in fact_rows:
    print(row)


## Results visualization

The closing figure uses small multiples for the per-rung artifact and one summary curve across D1-D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for index, rung in enumerate(fact_ladder):
    item = rung["items"][-1]
    result = claim_factuality(item["answer"], item["evidence"], item["confidences"], item["correctness"], force_atomic=True)
    colors = ["seagreen" if claim_supported(claim, item["evidence"]) else "tomato" for claim in result["claims"]]
    axes[0, index].bar(range(len(result["claims"])), [1] * len(result["claims"]), color=colors)
    axes[0, index].set_title(rung["name"])
    axes[0, index].set_xlabel("claims")

axes[1, 0].plot([row["rung"] for row in fact_rows], [row["metric"] for row in fact_rows], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_ylabel("unsupported rate")
axes[1, 0].set_title("unsupported vs rung")
for blank in axes[1, 1:]:
    blank.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Reproduce the named D5 failure, then turn on the fix and compare the behavior.

In [ ]:

d5_item = fact_ladder[-1]["items"][-1]
vague = claim_factuality(d5_item["answer"], d5_item["evidence"], d5_item["confidences"], d5_item["correctness"], force_atomic=False)
atomic = claim_factuality(d5_item["answer"], d5_item["evidence"], d5_item["confidences"], d5_item["correctness"], force_atomic=True)
print("vague extraction", vague["claims"], vague["unsupported_rate"])
print("atomic claims with citations required", atomic["claims"], atomic["unsupported_rate"])


## Evaluate it + Practice

- Compare the planned metric with a no-skill baseline such as answering without tools, retrieval, claims, intervals, or guardrails.
- Sanity-check one hand-computable lesson number before trusting the ladder.
- Ablate the key idea and verify the metric drops or the failure signal rises.
- Watch failure signals: retry loops, irrelevant evidence, hidden unsupported claims, noisy rank changes, or unsafe tool actions.

Practice prompts:
- Write an answer with one vague sentence hiding two claims, then force atomic extraction.
- Tune the abstention threshold from 0.5 to 0.9 and report coverage versus unsupported rate.
- Add one evidence snippet and show whether factuality improves or merely confidence improves.
